# 사용할 코드만 정리

In [ ]:
# 도구 불러오기
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from glob import glob
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.font_manager as fm

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [2]:
# 원본 메타데이터 로드
df_meta = pd.read_csv("../../dataset/metadata.csv")

In [3]:
# 전처리 (시간/수치형(Capacity, Re, Rct)/file_num 파생)

def clean_parse_time(x):
    if pd.isna(x):
        return pd.NaT
    
    try:
        s = str(x)
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if len(parts) < 3:
            return pd.NaT
        
        nums = [float(p) for p in parts]
        
        year   = int(nums[0])
        month  = int(nums[1])
        day    = int(nums[2])
        hour   = int(nums[3]) if len(nums) > 3 else 0
        minute = int(nums[4]) if len(nums) > 4 else 0
        
        # 핵심: round 제거
        second = int(nums[5]) if len(nums) > 5 else 0
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hour, minute=minute, second=second)
    
    except Exception as e:
        print(f"[파싱 실패] {x} | {e}")
        return pd.NaT

def clean_numeric(x):
    """수치형 데이터의 대괄호 제거 및 float 변환"""
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [전처리 실행] ---
df = df_meta.copy()

# 1. 시간 데이터 정제 (소수점 초까지 완벽 대응)
df['start_time'] = df['start_time'].apply(clean_parse_time)

# 2. 수치형 데이터 정제 (Capacity, Re, Rct)
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 물리적 순서 정렬 (파일 번호 기준)
df['file_num'] = df['filename'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(['battery_id', 'file_num']).reset_index(drop=True)

# 최종 결과 검증
print(f"전체 데이터 개수: {len(df):,}개")
print(f"시간 결측치(NaT): {df['start_time'].isna().sum()}개")
print(f"수치형 결측치(Cap): {df['Capacity'].isna().sum()}개")


전체 데이터 개수: 7,565개
시간 결측치(NaT): 0개
수치형 결측치(Cap): 4796개


In [4]:
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,file_num
0,charge,2008-04-02 13:08:17,24,B0005,0,5121,05121.csv,NaN,NaN,NaN,5121
1,discharge,2008-04-02 15:25:41,24,B0005,1,5122,05122.csv,1.856487,NaN,NaN,5122
2,charge,2008-04-02 16:37:51,24,B0005,2,5123,05123.csv,NaN,NaN,NaN,5123
3,discharge,2008-04-02 19:43:48,24,B0005,3,5124,05124.csv,1.846327,NaN,NaN,5124
4,charge,2008-04-02 20:55:40,24,B0005,4,5125,05125.csv,NaN,NaN,NaN,5125
...,...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:45,24,B0056,247,7309,07309.csv,NaN,0.102677,0.170394,7309
7561,discharge,2010-09-30 08:08:36,4,B0056,248,7310,07310.csv,1.137273,NaN,NaN,7310
7562,charge,2010-09-30 08:48:54,4,B0056,249,7311,07311.csv,NaN,NaN,NaN,7311
7563,discharge,2010-09-30 11:50:17,4,B0056,250,7312,07312.csv,1.129059,NaN,NaN,7312


In [ ]:
# 'B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data 구성하기
# SOH EOL RUL cycle 파생

# 대상 배터리 ID 리스트
target_battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']

# 데이터 저장할 폴더 경로
data_folder = "../../dataset/data"

# 데이터를 쌓아두기 위한 저장소(Dictionary) 생성
collected_data = {}


### 각 배터리별 EOL 사이클 번호를 먼저 파악 (RUL 계산용)
# SOH 80% 기준을 찾기 위해 메타데이터(df)에서 미리 계산
eol_dict = {}
for b_id in target_battery_ids:
    b_meta = df[(df['battery_id'] == b_id) & (df['type'] == 'discharge')].copy()
    if not b_meta.empty:
        initial_cap = b_meta['Capacity'].iloc[0] # 첫 번째 방전 용량
        # SOH 80% 이하인 첫 번째 행의 인덱스(순번) 찾기
        eol_idx = np.where((b_meta['Capacity'] / initial_cap) * 100 <= 80)[0]
        eol_dict[b_id] = eol_idx[0] + 1 if len(eol_idx) > 0 else np.nan

for battery_id in target_battery_ids:
    # 시간순 정렬 (매칭 오류 방지)
    filtered_df = df[df['battery_id'] == battery_id].sort_values('start_time')
    cycle_counters = {'charge': 1, 'discharge': 1, 'impedance': 1}
    
    # 기준 용량 정의 (해당 배터리의 첫 번째 방전 용량)
    dis_rows = filtered_df[filtered_df['type'] == 'discharge']
    first_cap = dis_rows['Capacity'].iloc[0] if not dis_rows.empty else None

    for _, row in filtered_df.iterrows():
        file_path = os.path.join(data_folder, row['filename'])

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)
            d_type = row['type']
            current_cycle = cycle_counters[d_type]

            # --- [1단계: 파생 변수 로직 정의] ---
            # SOH 정의: 오직 discharge 타입일 때만 해당 사이클의 용량으로 계산
            if d_type == 'discharge' and first_cap and pd.notnull(row['Capacity']):
                soh_val = (row['Capacity'] / first_cap) * 100
            else:
                soh_val = np.nan # 다른 타입은 우선 NaN 처리
            
            # EOL 정의: 미리 계산된 eol_dict 이용
            eol_val = eol_dict.get(battery_id)
            
            # RUL 정의: (사망 사이클 - 현재 사이클)
            rul_val = (eol_val - current_cycle) if pd.notnull(eol_val) else np.nan
            if pd.notnull(rul_val): rul_val = max(0, rul_val)

            # --- [2단계: 데이터프레임 열 추가] ---
            temp_df['start_time'] = row['start_time']
            temp_df['battery_id'] = battery_id
            temp_df['type'] = d_type
            temp_df['ambient_temperature'] = row['ambient_temperature']
            
            temp_df['cycle'] = current_cycle
            temp_df['SOH'] = soh_val  # 정의된 값 주입
            temp_df['EOL_cycle'] = eol_val
            temp_df['RUL'] = rul_val

            # 타입별 특화 데이터 추가
            if d_type == 'discharge':
                temp_df['Capacity'] = row['Capacity']
            elif d_type == 'impedance':
                temp_df['Re'] = row['Re']
                temp_df['Rct'] = row['Rct']

            # 카운터 증가 및 저장
            cycle_counters[d_type] += 1
            key = f"df_{d_type}_{battery_id}"
            if key not in collected_data:
                collected_data[key] = []
            collected_data[key].append(temp_df)

# --- [3단계: 데이터 통합 및 결측치 전파] ---
for key, df_list in collected_data.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # SOH 전파: discharge 파일들 사이의 간극(charge, impedance 등)을 메워줌
    # 단, 사용자님이 'discharge에만' 있길 원하신다면 이 섹션을 제외하면 됩니다.
    # 만약 모든 행에 SOH가 필요하다면 아래 ffill/bfill을 유지하세요.
    if 'SOH' in combined_df.columns:
        combined_df['SOH'] = combined_df['SOH'].ffill()
        combined_df['SOH'] = combined_df['SOH'].bfill()
    
    globals()[key] = combined_df
    print(f"✅ {key} 생성 완료 | 파일 {len(df_list)}개 통합 | 크기: {combined_df.shape} | SOH 결측치: {combined_df['SOH'].isnull().sum()} | cycle/SOH/EOL/RUL 파생 완료")


In [ ]:
# 대표로 하나 지정하여 null값 확인
df_discharge_B0005.info()

In [ ]:
# 대표로 하나 지정하여 null값 확인
df_charge_B0005.info()

In [ ]:
# 대표로 하나 지정하여 null값 확인
df_impedance_B0005.info()

In [ ]:
# 간단 확인
display(df_charge_B0006)
display(df_discharge_B0007)
display(df_impedance_B0005) 

# 분석

# discharge 분석

배터리 4개를 모두 합침


In [ ]:
# 1. 4개 배터리 데이터를 하나로 통합 (df_battery_discharge)

battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_dfs = []

for b_id in battery_ids:
    df_name = f"df_discharge_{b_id}"
    if df_name in globals():
        # .copy()를 써서 원본 데이터와 연결을 끊고 독립적인 복사본을 만들기
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id  # 배터리 식별자 추가
        all_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_discharge = pd.concat(all_dfs, ignore_index=True)

print(f"✅ discharge 통합 - 전체 행 개수: {len(df_battery_discharge)}")
print(f"포함된 배터리 id: {df_battery_discharge['battery_id'].unique()}")

In [ ]:
df_battery_discharge.columns

# charge 분석

배터리 4개를 모두 합침


In [ ]:
# 1. 충전(Charge) 데이터 통합
battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_charge_dfs = []

for b_id in battery_ids:
    df_name = f"df_charge_{b_id}"
    if df_name in globals():
        # 복사본 생성 및 ID 추가
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id
        all_charge_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_charge = pd.concat(all_charge_dfs, ignore_index=True)

print(f"charge 통합 - 전체 행 개수: {len(df_battery_charge)}")
print(f"포함된 배터리 id: {df_battery_charge['battery_id'].unique()}")

In [ ]:
df_battery_charge.columns

In [ ]:
# [EDA 1] 특정 배터리의 사이클별 Capacity 변화 시각화
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_battery_discharge, x='cycle', y='Capacity', hue='battery_id')
plt.title('Capacity Degradation by Battery ID')
plt.show()

# [Pre-processing] 특정 임계치 이하의 비정상적 Capacity 값 처리
# 예: 갑자기 0으로 떨어지거나 이전 값 대비 20% 이상 튀는 경우 제거
def clean_capacity(df):
    threshold = 0.1 # 예시 임계치
    df['Capacity'] = df['Capacity'].mask(df['Capacity'] < 1.0) # 비정상 저용량 제거
    df['Capacity'] = df['Capacity'].interpolate()
    return df

# [EDA 2] 충전 시 전압-전류 관계 (CC-CV 구간 확인)
sample_charge = df_battery_charge[(df_battery_charge['battery_id'] == 'B0005') & (df_battery_charge['cycle'] == 10)]
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(sample_charge['Time'], sample_charge['Voltage_measured'], 'g-')
ax2.plot(sample_charge['Time'], sample_charge['Current_measured'], 'b-')
ax1.set_ylabel('Voltage (V)', color='g')
ax2.set_ylabel('Current (A)', color='b')
plt.title('Charge Profile (Cycle 10)')
plt.show()

# impedance 분석

배터리 4개를 모두 합침

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 2개의 그래프(Re, Rct)를 상하로 배치
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12), sharex=True)

# 1. Re (Electrolyte Resistance) 비교
sns.lineplot(data=df_battery_impedance, x='cycle', y='Re', hue='battery_id', 
             marker='o', markersize=4, ax=ax1)
ax1.set_title('Electrolyte Resistance (Re) Comparison', fontsize=14)
ax1.set_ylabel('Resistance (Ohm)')
ax1.grid(True, linestyle='--', alpha=0.5)

# 2. Rct (Charge Transfer Resistance) 비교
sns.lineplot(data=df_battery_impedance, x='cycle', y='Rct', hue='battery_id', 
             marker='s', markersize=4, ax=ax2)
ax2.set_title('Charge Transfer Resistance (Rct) Comparison', fontsize=14)
ax2.set_ylabel('Resistance (Ohm)')
ax2.set_xlabel('Cycle')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 2. 임피던스(Impedance) 데이터 통합
battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_impedance_dfs = []

for b_id in battery_ids:
    df_name = f"df_impedance_{b_id}"
    if df_name in globals():
        # 복사본 생성 및 ID 추가
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id
        all_impedance_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_impedance = pd.concat(all_impedance_dfs, ignore_index=True)

print(f"Impedance 통합 - 전체 행 개수: {len(df_battery_impedance)}")
print(f"포함된 배터리 id: {df_battery_impedance['battery_id'].unique()}")

In [ ]:
df_battery_impedance

In [ ]:
df_battery_impedance.describe

In [ ]:
df_battery_impedance.describe()

In [ ]:
df_battery_impedance.info()

# 복소수 컬럼에서 유의한 컬럼 7개 파생

In [ ]:
# 1. 대상 원본 컬럼 리스트 (기존 변환된 complex 타입 기준)
complex_cols = ['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance']

# 2. 7개 핵심 파생 컬럼만 생성 (에러 방지용 타입 변환 추가)

# 문자열로 변해버린 컬럼을 다시 복소수(complex) 타입으로 변환하는 함수
def to_complex(x):
    if pd.isna(x) or x == '': return np.nan
    try:
        return complex(x) if isinstance(x, str) else x
    except:
        return np.nan

# (A) 전류 및 비율의 '크기(Magnitude)' 계산 전 타입 변환 적용
for col in ['Sense_current', 'Battery_current', 'Current_ratio']:
    df_battery_impedance[col] = df_battery_impedance[col].apply(to_complex)

df_battery_impedance['Sense_current_mag'] = df_battery_impedance['Sense_current'].apply(abs)
df_battery_impedance['Battery_current_mag'] = df_battery_impedance['Battery_current'].apply(abs)
df_battery_impedance['Current_ratio_mag'] = df_battery_impedance['Current_ratio'].apply(abs)

# (B) 임피던스 컬럼들도 동일하게 타입 변환 후 계산
for col in ['Battery_impedance', 'Rectified_Impedance']:
    df_battery_impedance[col] = df_battery_impedance[col].apply(to_complex)

df_battery_impedance['Battery_impedance_real'] = df_battery_impedance['Battery_impedance'].apply(lambda x: x.real if pd.notnull(x) else np.nan)
df_battery_impedance['Rectified_Impedance_real'] = df_battery_impedance['Rectified_Impedance'].apply(lambda x: x.real if pd.notnull(x) else np.nan)

# (C) 위상(Phase) 계산
df_battery_impedance['Battery_impedance_phase'] = df_battery_impedance['Battery_impedance'].apply(lambda x: np.angle(x, deg=True) if pd.notnull(x) else np.nan)
df_battery_impedance['Rectified_Impedance_phase'] = df_battery_impedance['Rectified_Impedance'].apply(lambda x: np.angle(x, deg=True) if pd.notnull(x) else np.nan)
# 3. 불필요하게 생성되었던 나머지 파생 컬럼 삭제
all_cols = df_battery_impedance.columns.tolist()
keep_cols = complex_cols + [
    'start_time', 'battery_id', 'type', 'ambient_temperature', 'cycle', 
    'SOH', 'EOL_cycle', 'RUL', 'Re', 'Rct',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase', 
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]
df_battery_impedance = df_battery_impedance[keep_cols]

# 결과 확인
print(df_battery_impedance.info())

df_battery_impedance.to_csv('df2_battery_impedance.csv', index=False, encoding='utf-8-sig')

In [ ]:
df_battery_impedance.head()

In [ ]:
# 수치형칼럼 describe()

# 1. ambient_temperature와 SOH를 제외한 11개 컬럼 정의
selected_cols = [
    'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase', 
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]

# 2. 요약 통계량 확인 (가로로 길게 보는 T 사용)
print("--- 11개 핵심 컬럼 요약 통계 ---")
display(df_battery_impedance[selected_cols].describe().T)

# 3. 실제 데이터 행(Row) 확인 (상위 5개)
print("\n--- 실제 데이터 행 확인 ---")
display(df_battery_impedance[selected_cols].head())

"""
물리적 오류 및 이상치 (전처리 필요)
Battery_impedance_real (min: -4.908): 저항의 최소값이 음수입니다. 물리적으로 저항이 0보다 작을 수는 없으므로, 이 행들은 센서 에러이거나 계산 오류입니다. (정상 범위는 보통 0.1~0.3 내외)
Battery_impedance_phase (min: -179.3, max: 179.4): 위상각이 180도 가까이 튀었습니다. 배터리 내부 화학 반응은 보통 -20° ~ 5° 사이에서 움직여야 합니다. 180도 차이가 난다는 건 신호가 완전히 뒤집힌(Inverted) 쓰레기 데이터라는 뜻입니다.
Current_ratio_mag (max: 14.25): 평균(2.62)에 비해 최대값이 너무 큽니다. 전류 비율이 14배까지 치솟았다는 것은 전류 센서 중 하나가 순간적으로 먹통이 되었을 가능성이 큽니다

배터리 노화 및 수명 지표 : 배터리의 상태를 나타내는 지표들의 분포입니다.
Re & Rct: Re(전해질 저항) 평균: 0.059, Rct(전하 전달 저항) 평균: 0.082입니다.
두 값 모두 표준편차(std)가 매우 작아 비교적 안정적으로 보이지만, Rct가 Re보다 평균적으로 높으므로 배터리 내부 화학 반응 저항이 지배적임을 알 수 있습니다.
cycle: 최소 1에서 최대 278까지 분포되어 있습니다. 배터리 하나가 수명을 다할 때까지의 전 과정을 담고 있는 데이터셋임을 보여줍니다.
RUL (Remaining Useful Life): 평균 19회 정도 남았으며, 75% 지점이 34회인 것으로 보아 수명이 얼마 남지 않은 배터리 데이터가 다수 포함되어 있습니다.

전류 센서 데이터 (_mag)
Sense_current_mag (834.4) vs Battery_current_mag (328.1):
센서에서 측정된 전류의 크기가 배터리에 흐르는 실전류보다 약 2.5배 정도 크게 찍힙니다.
이는 앞서 나온 Current_ratio_mag 평균인 2.62와 일치하며, 시스템의 전류 증폭 비율이 약 2.6배로 설정되어 있음을 시사합니다.

보정 데이터 (Rectified_Impedance)
 count (34,593): 다른 컬럼(42,576)에 비해 약 **8,000개 정도 데이터가 부족(NaN)**합니다. 이 부분은 결측치 처리를 하지 않기로 하셨으니, 분석 시 이 빈칸을 감안해야 합니다.
Rectified_Impedance_phase: 보정 전(Battery_impedance_phase)에는 -179~179로 날뛰던 위상값이 보정 후에는 -4.8 ~ 3.9 사이로 매우 안정되었습니다. 보정 알고리즘이 튀는 위상값을 잘 잡아낸 것으로 보입니다.
"""

# 내 지피티 ↓

구분	     -        전처리 전 (Raw)	      -       전처리 후 (Processed)

이상치	: 튀는 값에 의한 모델 왜곡 위험	>  Winsorize로 수치 안정화

위상각	: 산술 평균 시 물리적 오류 발생	>  Sin/Cos 변환으로 물리적 일관성 확보

변수 분포	: 쏠림 현상으로 학습 효율 저하	>  로그 변환으로 학습 최적화

데이터 누수	: EOL 정보 포함으로 가짜 성능 발생 >	순수 측정값과 Target의 엄격한 분리

In [ ]:
# 데이터 정제(Cleaning), 차원 변환(Transformation), 통계적 집계(Aggregation)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. 초기화 및 무결성 체크
def initialize_pipeline(df_raw):
    df = df_raw.copy()
    # 시간 변환 및 정렬
    if 'start_time' in df.columns:
        df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df = df.sort_values(['battery_id', 'cycle', 'start_time']).reset_index(drop=True)
    
    # point_idx 생성 (48행 구조 가정, 누락 대응 위해 cumcount 사용)
    df['point_idx'] = df.groupby(['battery_id', 'cycle']).cumcount()
    
    # RUL 공식 검증 (Target 생성)
    df['RUL_target'] = np.maximum(df['EOL_cycle'] - df['cycle'], 0)
    return df

# 2. 진단 단계 (VIF 및 상관분석)
def diagnose_data(df, feature_cols, target_col='RUL_target'):
    print("=== [STEP 2] 데이터 진단 시작 ===")
    diag_df = df[feature_cols + [target_col]].dropna()
    
    # 상관관계 분석
    corr_matrix = diag_df.corr()
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
    
    # VIF (다중공선성) 분석
    X = sm.add_constant(diag_df[feature_cols])
    vif_df = pd.DataFrame({
        'feature': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).sort_values('VIF', ascending=False)
    
    return target_corr, vif_df

# 3. 정제 단계 (Winsorize 및 변환)
def winsorize_by_group(df, col, group_cols, q=0.005):
    result = df[col].copy()
    # 그룹별로 계산하되 데이터가 너무 적으면 전역 기준 적용(Fallback)
    for name, group in df.groupby(group_cols):
        valid = group[col].dropna()
        if len(valid) > 10:
            low, high = valid.quantile(q), valid.quantile(1-q)
            result.loc[group.index] = group[col].clip(low, high)
    return result

def preprocess_features(df):
    print("=== [STEP 3] 피처 정제 및 생성 ===")
    
    # (1) Cycle-level 변수 정제 (Re, Rct)
    for col in ['Re', 'Rct']:
        df[f'{col}_clean'] = winsorize_by_group(df, col, ['battery_id'])
        
    # (2) Point-level 변수 정제
    point_cols = ['Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag', 'Battery_impedance_real']
    for col in point_cols:
        df[f'{col}_clean'] = winsorize_by_group(df, col, ['battery_id', 'point_idx'])
        
    # (3) 로그 변환 (Ratio)
    df['Current_ratio_mag_log1p'] = np.log1p(df['Current_ratio_mag_clean'])
    
    # (4) Phase 변환 (Sin/Cos) - 원형 데이터 왜곡 방지
    for col in ['Battery_impedance_phase', 'Rectified_Impedance_phase']:
        rad = np.deg2rad(df[col])
        df[f'{col}_sin'] = np.sin(rad)
        df[f'{col}_cos'] = np.cos(rad)
        
    return df

# 4. 집계 및 최종 Target 분리
def create_final_dataset(df):
    print("=== [STEP 4] 최종 사이클 단위 데이터셋 생성 ===")
    
    def circular_mean_deg(x):
        x = x.dropna()
        if len(x) == 0: return np.nan
        rad = np.deg2rad(x)
        return np.rad2deg(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean()))

    cycle_df = df.groupby(['battery_id', 'cycle']).agg({
        'RUL_target': 'first', # 정답(Y)
        'Re_clean': 'first',
        'Rct_clean': 'first',
        'Battery_impedance_real_clean': 'mean',
        'Battery_impedance_phase': circular_mean_deg,
        'Battery_impedance_phase_sin': 'mean',
        'Battery_impedance_phase_cos': 'mean',
        'Current_ratio_mag_log1p': 'mean'
    }).reset_index()
    
    return cycle_df

# --- 실행 파이프라인 ---
# 초기화
df_processed = initialize_pipeline(df_battery_impedance)

# 진단 (정제 전 원본 상태 확인)
features_to_check = ['Re', 'Rct', 'Battery_impedance_real', 'Battery_impedance_phase']
target_corr, vif_result = diagnose_data(df_processed, features_to_check)

print("\n[진단 결과: RUL과의 상관관계]\n", target_corr)
print("\n[진단 결과: 다중공선성(VIF)]\n", vif_result)

# 정제 및 변환
df_processed = preprocess_features(df_processed)

# 최종 학습용 데이터 생성
final_learning_df = create_final_dataset(df_processed)

print("\n=== 전처리 완료 ===")
print(f"최종 데이터 형태: {final_learning_df.shape}")

In [ ]:
def plot_winsorize_effect(df, col):
    plt.figure(figsize=(12, 5))
    
    # 전처리 전
    plt.subplot(1, 2, 1)
    sns.boxplot(y=df[col])
    plt.title(f'Before Winsorize: {col}')
    
    # 전처리 후
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[f'{col}_clean'])
    plt.title(f'After Winsorize: {col}_clean')
    
    plt.tight_layout()
    plt.show()

# 실행 예시 (가장 영향이 컸던 Re 확인)
plot_winsorize_effect(df_processed, 'Re')

In [ ]:
def plot_phase_transformation(df, phase_col):
    plt.figure(figsize=(12, 5))
    
    # 1. 원본 Phase 분포
    plt.subplot(1, 2, 1)
    sns.histplot(df[phase_col], bins=50, kde=True)
    plt.title('Original Phase Distribution (Deg)')
    
    # 2. Sin vs Cos 산점도 (단위 원 형태 확인)
    plt.subplot(1, 2, 2)
    plt.scatter(df[f'{phase_col}_cos'], df[f'{phase_col}_sin'], alpha=0.1)
    plt.xlabel('Cos Component')
    plt.ylabel('Sin Component')
    plt.title('Phase in Unit Circle Space')
    plt.axis('equal') # 원형 유지를 위해 축 비율 고정
    
    plt.tight_layout()
    plt.show()

# 실행 예시
plot_phase_transformation(df_processed, 'Battery_impedance_phase')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_preprocessing_results(df_proc):
    # 그래프 스타일 설정
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # [1] Winsorize 효과 (Re 컬럼 예시)
    # Re와 Re_clean 비교
    sns.boxplot(data=df_proc[['Re', 'Re_clean']], ax=axes[0, 0])
    axes[0, 0].set_title('1. Winsorize Effect (Internal Resistance: Re)', fontsize=13)
    axes[0, 0].set_ylabel('Resistance Value')

    # [2] Phase 변환 (Sin vs Cos 산점도)
    # 위상각이 단위 원(Unit Circle) 상에서 어떻게 분포하는지 확인
    axes[0, 1].scatter(df_proc['Battery_impedance_phase_cos'], 
                       df_proc['Battery_impedance_phase_sin'], 
                       alpha=0.2, color='green')
    axes[0, 1].set_title('2. Phase Transformation (Unit Circle)', fontsize=13)
    axes[0, 1].set_xlabel('Cos Component')
    axes[0, 1].set_ylabel('Sin Component')
    axes[0, 1].axis('equal') 

    # [3] Log 변환 효과 (Current_ratio_mag)
    # 에러 수정: 정확한 컬럼명 사용
    sns.kdeplot(df_proc['Current_ratio_mag'], fill=True, label='Original', ax=axes[1, 0])
    sns.kdeplot(df_proc['Current_ratio_mag_log1p'], fill=True, label='Log-transformed', ax=axes[1, 0])
    axes[1, 0].set_title('3. Log Transformation (Current Ratio)', fontsize=13)
    axes[1, 0].legend()

    # [4] 변수 간 상관관계 히트맵 (전처리 후)
    # 모델에 들어가는 핵심 피처들의 관계 확인
    features = ['Re_clean', 'Rct_clean', 'Battery_impedance_real_clean', 
                'Battery_impedance_phase_sin', 'Current_ratio_mag_log1p']
    sns.heatmap(df_proc[features].corr(), annot=True, cmap='RdBu_r', center=0, ax=axes[1, 1])
    axes[1, 1].set_title('4. Feature Correlation Matrix', fontsize=13)

    plt.tight_layout()
    plt.show()

# 실행
plot_preprocessing_results(df_processed)

In [ ]:
# 머신러닝

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 1. 학습용 피처(X)와 타깃(y) 분리
# 메타 데이터(id, cycle)와 정답(RUL_target)을 제외한 나머지를 학습에 사용
features = [
    'Re_clean', 'Rct_clean', 'Battery_impedance_real_clean',
    'Battery_impedance_phase_sin', 'Battery_impedance_phase_cos',
    'Current_ratio_mag_log1p'
]

X = final_learning_df[features]
y = final_learning_df['RUL_target']

# 2. 데이터 분할 (Battery_id 기준으로 나누는 것이 실제 예측 환경에 더 적합함)
# 여기서는 일반적인 무작위 분할을 예시로 함
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 모델 정의 및 학습 (CC 2A 환경의 비선형성 대응)
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# 4. 예측 및 평가
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"=== 모델 평가 결과 ===")
print(f"MAE (평균 절대 오차): {mae:.2f} cycles")
print(f"R² Score (설명력): {r2:.4f}")

# 5. 피처 중요도 확인 (어떤 물리적 인자가 수명에 큰 영향을 주었는가?)
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\n=== 피처 중요도 ===")
print(importances)

In [ ]:
# 모델성능 시각화 

# 1. Predicted vs Actual Plot
def plot_predicted_vs_actual(y_true, y_pred):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    plt.xlabel('Actual RUL (cycles)')
    plt.ylabel('Predicted RUL (cycles)')
    plt.title('Predicted vs. Actual RUL')
    plt.grid(True)
    plt.show()

# 2. Residual Plot
def plot_residual(y_true, y_pred):
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--', lw=2)
    plt.xlabel('Actual RUL (cycles)')
    plt.ylabel('Residuals (Actual - Predicted)')
    plt.title('Residual Plot')
    plt.grid(True)
    plt.show()

# 3. Battery-specific RUL Prediction Path
def plot_battery_rul_path(test_df, battery_id_col='battery_id'):
    # 테스트 데이터셋에서 배터리 아이디 가져오기
    unique_batteries = test_df[battery_id_col].unique()
    
    # 예시로 첫 번째 배터리 선택 (원하는 ID로 변경 가능)
    selected_battery = unique_batteries[0]
    battery_data = test_df[test_df[battery_id_col] == selected_battery].sort_values('cycle')
    
    plt.figure(figsize=(10, 6))
    plt.plot(battery_data['cycle'], battery_data['RUL_target'], 'k-', label='Actual RUL', lw=2)
    plt.plot(battery_data['cycle'], battery_data['RUL_pred'], 'r--', label='Predicted RUL', lw=2)
    plt.xlabel('Cycle')
    plt.ylabel('RUL (cycles)')
    plt.title(f'RUL Prediction Path for Battery: {selected_battery}')
    plt.legend()
    plt.grid(True)
    plt.show()

# --- 시각화 실행 ---

# 테스트 데이터셋에 예측값 추가 (시각화를 위해 필요)
# 주의: Group Split을 사용했다면 테스트에 사용된 battery_id를 포함한 원본 final_learning_df의 일부여야 함
# 여기서는 무작위 분할을 가정하고 X_test의 인덱스를 활용해 매칭
X_test_with_pred = final_learning_df.loc[X_test.index].copy()
X_test_with_pred['RUL_pred'] = y_pred

# 1. 예측값 vs 실제값
plot_predicted_vs_actual(y_test, y_pred)

# 2. 잔차 그래프
plot_residual(y_test, y_pred)

# 3. 특정 배터리의 수명 예측 경로 (battery_id 컬럼이 X_test에 포함되어 있어야 함)
# 만약 학습 시 battery_id를 뺐다면, 시각화 단계에서 다시 매칭해주는 작업이 필요
if 'battery_id' in X_test_with_pred.columns:
    plot_battery_rul_path(X_test_with_pred)
else:
    print("\n[주의] battery_id 컬럼이 테스트 데이터셋에 없어 배터리별 예측 경로 시각화를 건너뜁니다.")

# 내 지피티 ↑

## 유진님 지피티 ↓

In [ ]:
# =========================
# 0. 데이터 복사
# =========================
df_pre = df_battery_impedance.copy()

# 사용할 컬럼
selected_cols = [
    'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase',
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]

# 필수 메타 컬럼 확인
required_meta = ['battery_id', 'start_time']
missing_meta = [c for c in required_meta if c not in df_pre.columns]
if missing_meta:
    raise ValueError(f"필수 메타 컬럼 누락: {missing_meta}")

missing_selected = [c for c in selected_cols if c not in df_pre.columns]
if missing_selected:
    raise ValueError(f"선택 컬럼 누락: {missing_selected}")

# 시간형 변환
df_pre['start_time'] = pd.to_datetime(df_pre['start_time'], errors='coerce')

# 정렬
df_pre = df_pre.sort_values(['battery_id', 'cycle', 'start_time']).reset_index(drop=True)

# =========================
# 1. point_idx 생성
# =========================
# 이 CSV는 battery_id + cycle마다 48행 구조
# frequency 컬럼이 없으므로 cycle 내 순번을 point_idx로 사용
df_pre['point_idx'] = df_pre.groupby(['battery_id', 'cycle']).cumcount()

# =========================
# 2. 기본 무결성 체크
# =========================
# cycle / EOL / RUL / Re / Rct 는 cycle 내 상수인지 확인
const_check_cols = ['EOL_cycle', 'RUL', 'Re', 'Rct']
for col in const_check_cols:
    nunique_in_cycle = df_pre.groupby(['battery_id', 'cycle'])[col].nunique(dropna=False)
    bad_groups = nunique_in_cycle[nunique_in_cycle > 1]
    print(f"[{col}] cycle 내 상수 아닌 그룹 수: {len(bad_groups)}")

# RUL 공식 체크
df_pre['RUL_formula'] = np.maximum(df_pre['EOL_cycle'] - df_pre['cycle'], 0)
df_pre['flag_rul_mismatch'] = (df_pre['RUL'] != df_pre['RUL_formula'])

print("RUL 불일치 행 수:", df_pre['flag_rul_mismatch'].sum())

# =========================
# 3. 구조적 결측 플래그
# =========================
df_pre['flag_rectified_real_missing'] = df_pre['Rectified_Impedance_real'].isna().astype(int)
df_pre['flag_rectified_phase_missing'] = df_pre['Rectified_Impedance_phase'].isna().astype(int)

# 둘 다 존재하는지 여부
df_pre['Rectified_available'] = (
    df_pre['Rectified_Impedance_real'].notna() &
    df_pre['Rectified_Impedance_phase'].notna()
).astype(int)

# =========================
# 4. 그룹별 winsorize 함수
# =========================
def winsorize_by_group(df, col, group_cols, lower_q=0.005, upper_q=0.995):
    """
    원본은 보존하고, 그룹별 quantile 기준으로 clip한 Series 반환
    결측은 그대로 유지
    """
    result = pd.Series(index=df.index, dtype='float64')

    grouped = df.groupby(group_cols).groups

    for _, idx in grouped.items():
        s = df.loc[idx, col]
        valid = s.dropna()

        # 데이터가 너무 적으면 그대로 둠
        if len(valid) < 5:
            result.loc[idx] = s
            continue

        low = valid.quantile(lower_q)
        high = valid.quantile(upper_q)
        result.loc[idx] = s.clip(lower=low, upper=high)

    return result

# =========================
# 5. cycle-level 변수 전처리
# =========================
# Re, Rct : battery별 약한 클리핑
for col in ['Re', 'Rct']:
    clean_col = f'{col}_clean'
    df_pre[clean_col] = winsorize_by_group(
        df_pre, col, group_cols=['battery_id'], lower_q=0.005, upper_q=0.995
    )
    df_pre[f'flag_{col}_clipped'] = (
        df_pre[col].notna() &
        (df_pre[col] != df_pre[clean_col])
    ).astype(int)

# =========================
# 6. point-level 실수형 변수 전처리
# =========================
point_real_mag_cols = [
    'Sense_current_mag',
    'Battery_current_mag',
    'Current_ratio_mag',
    'Battery_impedance_real',
    'Rectified_Impedance_real'
]

for col in point_real_mag_cols:
    clean_col = f'{col}_clean'
    df_pre[clean_col] = winsorize_by_group(
        df_pre,
        col,
        group_cols=['battery_id', 'point_idx'],
        lower_q=0.005,
        upper_q=0.995
    )
    df_pre[f'flag_{col}_clipped'] = (
        df_pre[col].notna() &
        (df_pre[col] != df_pre[clean_col])
    ).astype(int)

# ratio는 로그 파생도 추가
df_pre['Current_ratio_mag_log1p'] = np.log1p(df_pre['Current_ratio_mag_clean'])

# =========================
# 7. phase 전처리 (중앙값 대체 X, sin/cos 변환)
# =========================
def add_phase_sin_cos(df, phase_col):
    rad_col = f'{phase_col}_rad'
    sin_col = f'{phase_col}_sin'
    cos_col = f'{phase_col}_cos'

    df[rad_col] = np.deg2rad(df[phase_col])
    df[sin_col] = np.sin(df[rad_col])
    df[cos_col] = np.cos(df[rad_col])
    return df

df_pre = add_phase_sin_cos(df_pre, 'Battery_impedance_phase')
df_pre = add_phase_sin_cos(df_pre, 'Rectified_Impedance_phase')

# phase 범위 체크용 flag (현재 데이터는 거의 이미 정상범위)
df_pre['flag_Battery_phase_out_of_range'] = (
    (df_pre['Battery_impedance_phase'] < -180) |
    (df_pre['Battery_impedance_phase'] > 180)
).astype(int)

df_pre['flag_Rectified_phase_out_of_range'] = (
    (df_pre['Rectified_Impedance_phase'] < -180) |
    (df_pre['Rectified_Impedance_phase'] > 180)
).astype(int)

# =========================
# 8. dense subset 생성
# =========================
# Rectified 계열은 point_idx 39~47이 구조적으로 결측
# 완전한 형태가 필요하면 point_idx < 39 subset 사용
df_dense39 = df_pre[df_pre['point_idx'] < 39].copy()

# =========================
# 9. cycle-level 집계용 함수
# =========================
def circular_mean_deg(x):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    rad = np.deg2rad(x)
    return np.rad2deg(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean()))

# cycle-level 요약표
cycle_df = (
    df_pre
    .groupby(['battery_id', 'cycle'])
    .agg(
        start_time=('start_time', 'first'),
        EOL_cycle=('EOL_cycle', 'first'),
        RUL=('RUL', 'first'),

        Re=('Re_clean', 'first'),
        Rct=('Rct_clean', 'first'),

        Sense_current_mag_mean=('Sense_current_mag_clean', 'mean'),
        Sense_current_mag_std=('Sense_current_mag_clean', 'std'),

        Battery_current_mag_mean=('Battery_current_mag_clean', 'mean'),
        Battery_current_mag_std=('Battery_current_mag_clean', 'std'),

        Current_ratio_mag_mean=('Current_ratio_mag_clean', 'mean'),
        Current_ratio_mag_std=('Current_ratio_mag_clean', 'std'),
        Current_ratio_mag_log1p_mean=('Current_ratio_mag_log1p', 'mean'),

        Battery_impedance_real_mean=('Battery_impedance_real_clean', 'mean'),
        Battery_impedance_real_std=('Battery_impedance_real_clean', 'std'),

        Rectified_Impedance_real_mean=('Rectified_Impedance_real_clean', 'mean'),
        Rectified_Impedance_real_std=('Rectified_Impedance_real_clean', 'std'),

        Battery_impedance_phase_cmean=('Battery_impedance_phase', circular_mean_deg),
        Rectified_Impedance_phase_cmean=('Rectified_Impedance_phase', circular_mean_deg),

        Battery_impedance_phase_sin_mean=('Battery_impedance_phase_sin', 'mean'),
        Battery_impedance_phase_cos_mean=('Battery_impedance_phase_cos', 'mean'),

        Rectified_Impedance_phase_sin_mean=('Rectified_Impedance_phase_sin', 'mean'),
        Rectified_Impedance_phase_cos_mean=('Rectified_Impedance_phase_cos', 'mean'),

        n_points=('point_idx', 'count'),
        rectified_available_points=('Rectified_available', 'sum'),

        n_Re_clipped=('flag_Re_clipped', 'sum'),
        n_Rct_clipped=('flag_Rct_clipped', 'sum')
    )
    .reset_index()
)

# =========================
# 10. 확인 출력
# =========================
print("=== 전처리 후 df_pre shape ===")
print(df_pre.shape)

print("\n=== dense subset (point_idx < 39) shape ===")
print(df_dense39.shape)

print("\n=== cycle_df shape ===")
print(cycle_df.shape)

print("\n=== 전처리 컬럼 일부 확인 ===")
display(df_pre[[
    'battery_id', 'cycle', 'point_idx',
    'Re', 'Re_clean',
    'Rct', 'Rct_clean',
    'Sense_current_mag', 'Sense_current_mag_clean',
    'Battery_current_mag', 'Battery_current_mag_clean',
    'Current_ratio_mag', 'Current_ratio_mag_clean', 'Current_ratio_mag_log1p',
    'Battery_impedance_real', 'Battery_impedance_real_clean',
    'Battery_impedance_phase', 'Battery_impedance_phase_sin', 'Battery_impedance_phase_cos',
    'Rectified_Impedance_real', 'Rectified_Impedance_real_clean',
    'Rectified_Impedance_phase', 'Rectified_Impedance_phase_sin', 'Rectified_Impedance_phase_cos',
    'Rectified_available'
]].head())

print("\n=== cycle_df 상위 5행 ===")
display(cycle_df)

## 유진님 지피티 ↑

In [ ]:
# 파생 칼럼에 대한 IQR

# 1. 시각화할 파생 컬럼 리스트
target_cols = [
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase', 
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:
df_battery_impedance.columns

In [ ]:
# 기본 칼럼에 대한 IQR

# 1. 시각화할 컬럼 리스트
target_cols = [
      'start_time', 'ambient_temperature',
      'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct'
]
selected_cols = [
    'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct',
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:
# 수치형의 IQR 확인

# 1. 시각화가 가능한 수치형(float, int) 컬럼만 선택
# 복소수(complex)는 절대값(abs) 처리를 해야 시각화가 가능하므로 제외하거나 변환 후 포함해야 함
numeric_cols = df_battery_impedance.select_dtypes(include=['float64', 'int64']).columns.tolist()

# 2. 서브플롯 설정 (컬럼 개수에 맞춰 행/열 계산)
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

plt.figure(figsize=(15, n_rows * 4))

for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    # 결측치(NaN)를 제외하고 박스 플롯 그리기
    sns.boxplot(y=df_battery_impedance[col].dropna())
    plt.title(f'Boxplot of {col}')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 시각화할 9개 컬럼 리스트 (기존 7개 + Re, Rct)
plot_cols = [
    'Re', 'Rct', 'Battery_impedance_real', 'Battery_impedance_phase',
    'Rectified_Impedance_real', 'Rectified_Impedance_phase',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag'
]

# 그래프 설정 (3행 3열)
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Battery Parameters Trend over Cycles (Clean Data)', fontsize=20)

axes = axes.flatten()

for i, col in enumerate(plot_cols):
    # 산점도(Scatter)로 추세 확인 (점 크기 조절 s=5)
    sns.scatterplot(data=df_battery_impedance, x='cycle', y=col, 
                    ax=axes[i], s=5, alpha=0.4, color='indianred')
    
    # 추세선(Trendline) 추가 - 데이터의 흐름을 더 잘 보게 해줍니다.
    sns.regplot(data=df_battery_impedance, x='cycle', y=col, 
                ax=axes[i], scatter=False, color='royalblue')
    
    axes[i].set_title(f'Cycle vs {col}', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# 시간에 따른 임피던스(저항) 변화 확인
plt.figure(figsize=(12, 6))

# 1. 원본 임피던스의 실수부 (저항)
plt.plot(df_battery_impedance['start_time'], 
         df_battery_impedance['Battery_impedance_real'], 
         label='Raw Impedance (Real)', 
         alpha=0.5, 
         color='blue')

# 2. 보정된 임피던스의 실수부 (Rectified)
plt.plot(df_battery_impedance['start_time'], 
         df_battery_impedance['Rectified_Impedance_real'], 
         label='Rectified Impedance', 
         color='red', 
         linewidth=1.5)

plt.title('Battery Impedance Trend Over Time', fontsize=14)
plt.xlabel('Time')
plt.ylabel('Resistance (Ohm)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 임피던스 전처리

In [ ]:
# 전처리 하기

# df_battery_impedance를 독립된 데이터프레임으로 확실히 만듭니다.
df_battery_impedance = df_battery_impedance.copy()

# 'is_valid'라는 체크용 컬럼 생성 (기본값 True)
df_battery_impedance['is_valid'] = True

# 1. 저항이 음수인 경우 (물리적 오류)
df_battery_impedance.loc[df_battery_impedance['Battery_impedance_real'] <= 0, 'is_valid'] = False

# 2. 위상각이 너무 튀는 경우 (측정 오류)
# 배터리 특성상 -45도 ~ 10도 범위를 벗어나면 신뢰하기 어렵습니다.
df_battery_impedance.loc[(df_battery_impedance['Battery_impedance_phase'] < -45) | 
                         (df_battery_impedance['Battery_impedance_phase'] > 20), 'is_valid'] = False

print(f"정상 데이터: {df_battery_impedance['is_valid'].sum()}개")
print(f"이상 데이터(보류): {(~df_battery_impedance['is_valid']).sum()}개")

In [ ]:
# 보정 데이터가 있는 것만 따로 보기 (원본 유지)
df_rectified_only = df_battery_impedance[df_battery_impedance['Rectified_Impedance'].notnull()]

In [ ]:
df_battery_impedance

In [ ]:
display(df_battery_impedance[selected_cols].describe().T)

In [ ]:
df_battery_impedance_clean.info()

In [ ]:
# 파생 칼럼에 대한 IQR

# 1. 시각화할 파생 컬럼 리스트
target_cols = [
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase', 
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:
# 기본 칼럼에 대한 IQR

# 1. 시각화할 컬럼 리스트
target_cols = [
      'start_time', 'ambient_temperature',
      'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct'
]
selected_cols = [
    'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct',
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:


# 시각화할 9개 컬럼 리스트 (기존 7개 + Re, Rct)
plot_cols = [
    'Re', 'Rct', 'Battery_impedance_real', 'Battery_impedance_phase',
    'Rectified_Impedance_real', 'Rectified_Impedance_phase',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag'
]

# 그래프 설정 (3행 3열)
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Battery Parameters Trend over Cycles (Clean Data)', fontsize=20)

axes = axes.flatten()

for i, col in enumerate(plot_cols):
    # 산점도(Scatter)로 추세 확인 (점 크기 조절 s=5)
    sns.scatterplot(data=df_battery_impedance, x='cycle', y=col, 
                    ax=axes[i], s=5, alpha=0.4, color='indianred')
    
    # 추세선(Trendline) 추가 - 데이터의 흐름을 더 잘 보게 해줍니다.
    sns.regplot(data=df_battery_impedance, x='cycle', y=col, 
                ax=axes[i], scatter=False, color='royalblue')
    
    axes[i].set_title(f'Cycle vs {col}', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# 진짜 깨끗한 데이터셋 (분석용 view)
df_battery_impedance_clean = df_battery_impedance[
    (df_battery_impedance['is_valid'] == True) & 
    (df_battery_impedance['Rectified_Impedance'].notnull())
]

# 이제 이 clean_view로 describe()를 해보세요. 수치가 훨씬 정상적으로 변할 겁니다.
display(df_battery_impedance_clean[selected_cols].describe().T)

In [ ]:
df_battery_impedance_clean

In [ ]:
df_battery_impedance_clean.info()

In [ ]:
# 파생 칼럼에 대한 IQR

# 1. 시각화할 파생 컬럼 리스트
target_cols = [
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag',
    'Battery_impedance_real', 'Battery_impedance_phase', 
    'Rectified_Impedance_real', 'Rectified_Impedance_phase'
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance_clean[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:
# 기본 칼럼에 대한 IQR

# 1. 시각화할 컬럼 리스트
target_cols = [
      'start_time', 'ambient_temperature',
      'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct'
]
selected_cols = [
    'cycle', 'EOL_cycle', 'RUL', 'Re', 'Rct',
]

# 2. 그래프 레이아웃 설정 (2행 4열)
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten() # 2D 배열을 1D로 변환하여 반복문 사용 용이하게 함

for i, col in enumerate(target_cols):
    # 박스플롯 그리기 (결측치는 자동으로 제외됨)
    sns.boxplot(y=df_battery_impedance_clean[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'IQR: {col}', fontsize=12)
    axes[i].set_ylabel('Value')
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

# 사용하지 않는 8번째 서브플롯은 삭제
fig.delaxes(axes[7])

plt.tight_layout()
plt.show()

In [ ]:


# 시각화할 9개 컬럼 리스트 (기존 7개 + Re, Rct)
plot_cols = [
    'Re', 'Rct', 'Battery_impedance_real', 'Battery_impedance_phase',
    'Rectified_Impedance_real', 'Rectified_Impedance_phase',
    'Sense_current_mag', 'Battery_current_mag', 'Current_ratio_mag'
]

# 그래프 설정 (3행 3열)
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Battery Parameters Trend over Cycles (Clean Data)', fontsize=20)

axes = axes.flatten()

for i, col in enumerate(plot_cols):
    # 산점도(Scatter)로 추세 확인 (점 크기 조절 s=5)
    sns.scatterplot(data=df_battery_impedance_clean, x='cycle', y=col, 
                    ax=axes[i], s=5, alpha=0.4, color='indianred')
    
    # 추세선(Trendline) 추가 - 데이터의 흐름을 더 잘 보게 해줍니다.
    sns.regplot(data=df_battery_impedance_clean, x='cycle', y=col, 
                ax=axes[i], scatter=False, color='royalblue')
    
    axes[i].set_title(f'Cycle vs {col}', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

3가지 EDA

In [12]:
print("d")

d
